# Bootstrapping em Machine Learning

## Frase a explicar
> **"Leading machine learning algorithms including random forests and gradient boosting use bootstrapping in order to make predictions."**

Este notebook explica essa ideia de forma didática, com **dados sintéticos** e **Python**.

## Objetivo
Ao final, você deve entender:

- o que é **bootstrapping**
- por que ele ajuda em machine learning
- como isso aparece em **Random Forest**
- em que sentido isso aparece ou **não aparece da mesma forma** em **Gradient Boosting**

## Resumo conceitual
**Bootstrapping** é uma técnica de reamostragem em que criamos várias amostras a partir do conjunto de treino, **com reposição**.

Isso significa que:

- uma mesma observação pode aparecer várias vezes na amostra
- algumas observações podem nem aparecer

A ideia central é gerar várias versões ligeiramente diferentes do dataset para estimar variabilidade e treinar modelos mais robustos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import resample

np.random.seed(42)

## 1. Criando um conjunto de dados

Vamos gerar um dataset de classificação binária parecido com um problema real:

- variáveis numéricas
- algum ruído
- separação imperfeita entre classes

Pense nisso como um problema do tipo:

- churn
- fraude
- resposta a campanha
- conversão


In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_clusters_per_class=2,
    class_sep=1.1,
    flip_y=0.04,
    random_state=42
)

feature_names = [f"x{i}" for i in range(1, 9)]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

df.head()

In [ ]:
df["target"].value_counts(normalize=True).rename("proportion")

## 2. O que é bootstrapping na prática?

Vamos pegar um dataset pequeno só para enxergar a mecânica.

Suponha que temos 10 linhas numeradas de 0 a 9.  
Uma amostra bootstrap sorteia **10 linhas com reposição**.

Então:

- algumas linhas aparecem repetidas
- outras ficam de fora


In [ ]:
small_index = np.arange(10)
bootstrap_sample = resample(small_index, replace=True, n_samples=len(small_index), random_state=42)

print("Índices originais :", small_index)
print("Amostra bootstrap :", bootstrap_sample)

In [ ]:
pd.Series(bootstrap_sample).value_counts().sort_index()

### Interpretação

Se uma linha aparece 2 ou 3 vezes, ela teve mais peso naquela amostra bootstrap.

Se uma linha não aparece, ela ficou **fora da amostra**.

Isso cria datasets ligeiramente diferentes, e isso é útil porque muitos algoritmos baseados em árvores são sensíveis a pequenas mudanças nos dados.


## 3. Exemplo manual: várias árvores treinadas em amostras bootstrap

Antes de usar Random Forest, vamos reproduzir a intuição manualmente:

1. criar várias amostras bootstrap do treino
2. treinar uma árvore em cada amostra
3. comparar as previsões

Isso mostra por que o bootstrap aumenta diversidade entre modelos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

trees = []
pred_probas = []

for seed in range(10):
    X_boot, y_boot = resample(X_train, y_train, replace=True, n_samples=len(X_train), random_state=seed)
    tree = DecisionTreeClassifier(max_depth=4, random_state=seed)
    tree.fit(X_boot, y_boot)
    trees.append(tree)
    pred_probas.append(tree.predict_proba(X_test)[:, 1])

pred_matrix = np.column_stack(pred_probas)
pred_matrix[:5, :5]

Cada coluna acima é a probabilidade prevista por uma árvore diferente.

Se o bootstrap não gerasse diversidade, todas as colunas seriam quase idênticas.  
Mas como cada árvore viu uma versão diferente dos dados, as previsões mudam.


In [ ]:
mean_pred = pred_matrix.mean(axis=1)
manual_ensemble_pred = (mean_pred >= 0.5).astype(int)

manual_acc = accuracy_score(y_test, manual_ensemble_pred)
manual_auc = roc_auc_score(y_test, mean_pred)

print(f"Ensemble manual com bootstrap - Accuracy: {manual_acc:.3f}")
print(f"Ensemble manual com bootstrap - ROC AUC : {manual_auc:.3f}")

## 4. Por que isso ajuda?

Uma árvore individual costuma ter:

- **baixa bias**: consegue aprender relações complexas
- **alta variância**: pequenas mudanças nos dados podem mudar muito a árvore

Quando fazemos várias árvores com bootstrap e depois agregamos as previsões:

- o erro aleatório tende a se cancelar
- a variância cai
- a generalização melhora

Isso é a lógica de **bagging** (*Bootstrap Aggregating*).


## 5. Random Forest: bootstrap de forma explícita

O **Random Forest** é uma coleção de árvores de decisão onde:

1. cada árvore é treinada em uma amostra bootstrap do treino
2. em cada divisão da árvore, só um subconjunto aleatório de variáveis é considerado

Esses dois mecanismos criam diversidade:

- bootstrap nas linhas
- aleatoriedade nas colunas/features


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    bootstrap=True,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print(f"Random Forest - Accuracy: {rf_acc:.3f}")
print(f"Random Forest - ROC AUC : {rf_auc:.3f}")

### Interpretação

Aqui o bootstrap está no coração do algoritmo.

Cada árvore do Random Forest vê um dataset diferente.  
Depois o modelo combina as previsões por:

- **voto majoritário** em classificação
- **média** em regressão

Ou seja, **Random Forest usa bootstrapping de forma direta e central**.


## 6. Visualizando a estabilidade do ensemble

Vamos comparar as previsões de várias árvores individuais com a média do ensemble.


In [ ]:
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.plot(pred_matrix[:50, i], alpha=0.7, label=f"Árvore {i+1}")
plt.plot(mean_pred[:50], linewidth=3, label="Média do ensemble")
plt.title("Probabilidades previstas: árvores individuais vs média")
plt.xlabel("Observações de teste")
plt.ylabel("Probabilidade prevista")
plt.legend()
plt.show()

A linha da média costuma ser menos errática do que as árvores individuais.  
Esse é exatamente o efeito esperado da agregação após bootstrap.


## 7. E o Gradient Boosting?

Aqui é importante fazer uma correção conceitual fina.

### O que o texto quis dizer
Muitos materiais didáticos agrupam vários algoritmos ensemble e dizem que eles "usam bootstrapping".

### O que é mais preciso dizer
- **Random Forest / Bagging**: usam bootstrapping de forma clássica e explícita
- **Gradient Boosting**: normalmente **não depende de bootstrap clássico da mesma forma**

No Gradient Boosting, o mecanismo principal é outro:

1. treina-se um modelo fraco
2. mede-se o erro/resíduo
3. treina-se o próximo modelo para corrigir o erro do anterior
4. repete-se esse processo sequencialmente

Ou seja, o ganho principal vem de **aprendizado sequencial dos erros**, não de bootstrap.

### Então por que às vezes associam boosting a bootstrapping?
Porque algumas variantes usam:

- **subsampling** das linhas
- às vezes chamado de **stochastic gradient boosting**

Mas isso não é exatamente o mesmo papel central que bootstrap tem no Random Forest.


In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

gb_acc = accuracy_score(y_test, gb_pred)
gb_auc = roc_auc_score(y_test, gb_proba)

print(f"Gradient Boosting - Accuracy: {gb_acc:.3f}")
print(f"Gradient Boosting - ROC AUC : {gb_auc:.3f}")

## 8. Comparação rápida

Vamos colocar os resultados lado a lado.


In [ ]:
results = pd.DataFrame({
    "model": ["Ensemble manual com bootstrap", "Random Forest", "Gradient Boosting"],
    "accuracy": [manual_acc, rf_acc, gb_acc],
    "roc_auc": [manual_auc, rf_auc, gb_auc]
})

results

## 9. Intuição final conectando com a frase original

### Parte correta da frase
A ideia geral de que **algoritmos ensemble usam reamostragem / múltiplos subconjuntos dos dados para produzir previsões mais robustas** está no caminho certo.

### Versão tecnicamente mais precisa
Uma formulação melhor seria:

> **Random forests rely heavily on bootstrap sampling to train many diverse trees and aggregate their predictions. Gradient boosting, in contrast, mainly improves predictions by sequentially correcting previous errors, although some variants use subsampling of the training data.**

Em português:

> **Random Forest depende fortemente de amostras bootstrap para treinar várias árvores diferentes e agregar suas previsões. Já o Gradient Boosting melhora as previsões principalmente ao corrigir sequencialmente os erros anteriores, embora algumas variantes usem subamostragem dos dados.**


## 10. Conclusão

### O que você deve guardar
- **Bootstrapping** = reamostrar com reposição
- isso gera datasets diferentes a partir do treino original
- **Random Forest** usa isso diretamente para reduzir variância
- **Gradient Boosting** não tem o bootstrap como mecanismo central; seu núcleo é a correção sequencial de erros
- alguns métodos de boosting usam subamostragem, mas isso é diferente de dizer que todo boosting funciona por bootstrap

### Leitura estatística da ideia
O bootstrap ajuda a:

- medir variabilidade
- reduzir instabilidade de modelos
- construir ensembles mais robustos
- melhorar generalização


## 11. Exercícios sugeridos

1. Troque `max_depth=4` por `max_depth=None` nas árvores e veja como muda a variância.
2. Aumente `n_estimators` no Random Forest e observe se o desempenho estabiliza.
3. Gere um dataset com mais ruído (`flip_y`) e veja qual modelo sofre mais.
4. Teste regressão em vez de classificação para observar o mesmo princípio.
5. Pesquise `subsample` em boosting e compare com bootstrap clássico.
